In [2]:
import numpy as np
import pandas as pd
import joblib
import os

from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE


df = pd.read_csv(r"C:\Users\hp\Downloads\customer_churn_data.csv")


df.drop(columns=["CustomerID"], inplace=True)
df["InternetService"] = df["InternetService"].fillna("NA")


# One-hot encode nominal columns
encoded = pd.get_dummies(
    df[["Gender", "InternetService", "TechSupport", "Churn"]],
    drop_first=True,
    dtype=int
)
df = pd.concat([df, encoded], axis=1)
df.drop(columns=["Gender", "InternetService", "TechSupport", "Churn"], inplace=True)


# Ordinal encode ContractType (ordered: Month-to-Month < One-Year < Two-Year)
ord_enc = OrdinalEncoder(categories=[["Month-to-Month", "One-Year", "Two-Year"]], dtype=int)
df["ContractType"] = ord_enc.fit_transform(df[["ContractType"]])



Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
UB = Q3 + 1.5 * IQR
LB = Q1 - 1.5 * IQR

df = df[
    (df["TotalCharges"] < UB["TotalCharges"]) & (df["TotalCharges"] > LB["TotalCharges"]) &
    (df["Age"] < UB["Age"]) & (df["Age"] > LB["Age"])
]
print(f"    Shape after outlier removal: {df.shape}")


X = df.drop(columns=["Churn_Yes"])
y = df["Churn_Yes"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selector.fit(X_train, y_train)

feature_imp = pd.DataFrame({
    "feature": X_train.columns,
    "imp_value": rf_selector.feature_importances_
}).sort_values(by="imp_value", ascending=False)

print(feature_imp.to_string(index=False))

selected_cols = feature_imp[feature_imp["imp_value"] > 0.05]["feature"].to_list()
print(f"    Selected features: {selected_cols}")

X_train = X_train[selected_cols]
X_test  = X_test[selected_cols]



# SMOTE — balance classes on training data only
smt = SMOTE(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

# StandardScaler — fit on resampled train, transform both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train_res)

y_pred = lr_model.predict(X_test_scaled)


print(f"  Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"  F1 Score  : {f1_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred))

# Save artifacts using joblib
os.makedirs("artifacts", exist_ok=True)
joblib.dump(lr_model,       "artifacts/model.pkl")
joblib.dump(scaler,         "artifacts/scaler.pkl")
joblib.dump(selected_cols,  "artifacts/selected_features.pkl")

print("Artifacts saved to /artifacts/")
print("   → artifacts/model.pkl")
print("   → artifacts/scaler.pkl")
print("   → artifacts/selected_features.pkl")


    Shape after outlier removal: (935, 10)
                    feature  imp_value
               ContractType   0.268555
            TechSupport_Yes   0.215107
                     Tenure   0.191715
             MonthlyCharges   0.147247
               TotalCharges   0.100345
         InternetService_NA   0.041551
                        Age   0.018822
InternetService_Fiber Optic   0.011242
                Gender_Male   0.005415
    Selected features: ['ContractType', 'TechSupport_Yes', 'Tenure', 'MonthlyCharges', 'TotalCharges']
  Accuracy  : 0.8449
  Precision : 1.0000
  Recall    : 0.8253
  F1 Score  : 0.9043

              precision    recall  f1-score   support

           0       0.42      1.00      0.59        21
           1       1.00      0.83      0.90       166

    accuracy                           0.84       187
   macro avg       0.71      0.91      0.75       187
weighted avg       0.93      0.84      0.87       187

Artifacts saved to /artifacts/
   → artifacts/model.